# ATDL Phase 2 — Annotation Preprocessing and YOLO Conversion

The raw source path is data/raw/NEU-DET/NEU-DET. This notebook verifies the existing Phase 2 outputs first and preserves the current split/artifacts by default. Set REBUILD_PHASE2 = True only when you intentionally want to regenerate processed data from the raw images and XML annotations.

The source coordinate convention is the verified Pascal VOC one-based inclusive convention recorded in the saved Phase 2 summary.

## 1. Find project paths and confirm the source dataset

In [1]:
from pathlib import Path
import hashlib
import json
import shutil
import xml.etree.ElementTree as ET
from collections import Counter
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "requirements-local.txt").is_file() and (p / "data").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open the atdl_final_project folder in VS Code.")
RAW_DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "NEU-DET" / "NEU-DET"
DATA_ROOT = PROJECT_ROOT / "data"
PROCESSED_ROOT = DATA_ROOT / "processed"
RESULTS_ROOT = PROJECT_ROOT / "results"
PHASE2_ROOT = RESULTS_ROOT / "phase2"
TABLES_DIR = RESULTS_ROOT / "tables"
CONFIG_ROOT = PROJECT_ROOT / "configs"
PHASE2_ROOT.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_ROOT.mkdir(parents=True, exist_ok=True)
print("Correct raw dataset path:", RAW_DATA_ROOT)
for required in (RAW_DATA_ROOT / "train" / "images", RAW_DATA_ROOT / "train" / "annotations",
                 RAW_DATA_ROOT / "validation" / "images", RAW_DATA_ROOT / "validation" / "annotations"):
    if not required.is_dir():
        raise FileNotFoundError(required)


Correct raw dataset path: ./data/raw/NEU-DET/NEU-DET


## 2. Fixed class mapping and reproducible settings

In [2]:
CLASS_NAMES = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]
CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}
CLASS_KEY_TO_ID = {name.lower().replace("-", "_"): idx for idx, name in enumerate(CLASS_NAMES)}
SEED = 42
COORDINATE_MODE = "voc_1_based_inclusive"
REBUILD_PHASE2 = False  # Preserve current processed files and split unless intentionally changed.
SPLIT_MANIFEST = DATA_ROOT / "splits" / "phase2_image_split.csv"
CONVERSION_MANIFEST = PHASE2_ROOT / "yolo_conversion_manifest.csv"
SUMMARY_PATH = PHASE2_ROOT / "phase2_dataset_summary.json"
DATA_YAML = CONFIG_ROOT / "neu_det_yolo.yaml"
print("Class IDs:", CLASS_TO_ID)
print("Coordinate mode:", COORDINATE_MODE)
print("Rebuild existing processed data:", REBUILD_PHASE2)


Class IDs: {'crazing': 0, 'inclusion': 1, 'patches': 2, 'pitted_surface': 3, 'rolled-in_scale': 4, 'scratches': 5}
Coordinate mode: voc_1_based_inclusive
Rebuild existing processed data: False


## 3. Discover source images and Pascal VOC annotations

In [3]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp"}
source_images = sorted(p for p in RAW_DATA_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)
xml_paths = sorted(RAW_DATA_ROOT.rglob("*.xml"))
xml_by_stem = {}
for path in xml_paths:
    xml_by_stem.setdefault(path.stem, []).append(path)
image_by_stem = {}
for path in source_images:
    image_by_stem.setdefault(path.stem, []).append(path)

duplicates = {stem: paths for stem, paths in image_by_stem.items() if len(paths) != 1}
ambiguous_xml = {stem: paths for stem, paths in xml_by_stem.items() if len(paths) != 1}
missing_xml = [path for path in source_images if path.stem not in xml_by_stem]
if duplicates or ambiguous_xml or missing_xml:
    raise ValueError(f"Ambiguous/missing source matches: images={len(duplicates)}, XML={len(ambiguous_xml)}, no XML={len(missing_xml)}")

source_rows = []
for image_path in source_images:
    xml_path = xml_by_stem[image_path.stem][0]
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    width = int(size.findtext("width")) if size is not None else 0
    height = int(size.findtext("height")) if size is not None else 0
    boxes = []
    for obj in root.findall("object"):
        name = (obj.findtext("name") or "").strip().lower().replace(" ", "_").replace("-", "_")
        if name not in CLASS_KEY_TO_ID:
            raise ValueError(f"Unknown class {name!r} in {xml_path}")
        bnd = obj.find("bndbox")
        if bnd is None:
            raise ValueError(f"Missing bndbox in {xml_path}")
        coords = {key: float(bnd.findtext(key)) for key in ("xmin", "ymin", "xmax", "ymax")}
        if width <= 0 or height <= 0:
            raise ValueError(f"Invalid image dimensions in {xml_path}")
        if not (1 <= coords["xmin"] <= coords["xmax"] <= width and 1 <= coords["ymin"] <= coords["ymax"] <= height):
            raise ValueError(f"Invalid VOC coordinates in {xml_path}: {coords}")
        boxes.append((CLASS_KEY_TO_ID[name], coords))
    if not boxes:
        raise ValueError(f"No defect boxes in {xml_path}")
    source_rows.append({
        "image": image_path, "xml": xml_path, "width": width, "height": height, "boxes": boxes,
        "image_sha256": hashlib.sha256(image_path.read_bytes()).hexdigest(),
    })
print(f"Matched {len(source_rows)} images to {len(xml_paths)} XML annotations.")
image_hash_groups = {}
for row in source_rows:
    image_hash_groups.setdefault(row["image_sha256"], []).append(row)
exact_duplicate_groups = {digest: rows for digest, rows in image_hash_groups.items() if len(rows) > 1}
def annotation_signature(row):
    return tuple(sorted(
        (class_id, tuple((key, round(float(coords[key]), 6)) for key in ("xmin", "ymin", "xmax", "ymax")))
        for class_id, coords in row["boxes"]
    ))

canonical_source_rows = []
duplicate_rows = []
for digest, rows in sorted(image_hash_groups.items()):
    ordered_rows = sorted(rows, key=lambda row: row["image"].relative_to(RAW_DATA_ROOT).as_posix())
    canonical_source_rows.append(ordered_rows[0])
    if len(ordered_rows) > 1:
        signatures = [annotation_signature(row) for row in ordered_rows]
        duplicate_rows.append({
            "sha256": digest, "duplicate_group_id": digest[:16], "image_count": len(ordered_rows),
            "canonical_path": ordered_rows[0]["image"].relative_to(RAW_DATA_ROOT).as_posix(),
            "excluded_duplicate_paths": "|".join(row["image"].relative_to(RAW_DATA_ROOT).as_posix() for row in ordered_rows[1:]),
            "annotations_identical": len(set(signatures)) == 1,
            "relative_paths": "|".join(row["image"].relative_to(RAW_DATA_ROOT).as_posix() for row in ordered_rows),
        })
duplicate_report = pd.DataFrame(duplicate_rows,
    columns=["sha256", "duplicate_group_id", "image_count", "canonical_path",
             "excluded_duplicate_paths", "annotations_identical", "relative_paths"])
duplicate_report.to_csv(TABLES_DIR / "exact_duplicate_groups.csv", index=False)
print(f"Matched {len(source_rows)} raw images; modeling dataset will use {len(canonical_source_rows)} unique image contents.")
print("Exact duplicate groups:", len(duplicate_report),
      "| duplicate images:", int(duplicate_report["image_count"].sum()) if not duplicate_report.empty else 0,
      "| groups with inconsistent annotations:", int((~duplicate_report["annotations_identical"]).sum()) if not duplicate_report.empty else 0)


Matched 1800 images to 1800 XML annotations.
Matched 1800 raw images; modeling dataset will use 1799 unique image contents.
Exact duplicate groups: 1 | duplicate images: 2 | groups with inconsistent annotations: 1


## 4. Reuse the existing split; stratify only if no manifest exists

In [4]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

source_keys = {str(row["image"].resolve()) for row in source_rows}
reused_existing_split = False
split_frame = None
assignments = {}
duplicate_split_conflicts = []

if SPLIT_MANIFEST.is_file() and not REBUILD_PHASE2:
    saved = pd.read_csv(SPLIT_MANIFEST)
    if {"image_path", "split"}.issubset(saved.columns) and not saved["image_path"].duplicated().any():
        split_by_path = {str(Path(row.image_path).resolve()): str(row.split)
                         for row in saved.itertuples(index=False)}
        if set(split_by_path) == source_keys and set(split_by_path.values()).issubset({"train", "val", "test"}):
            for digest, group_rows in exact_duplicate_groups.items():
                group_splits = {split_by_path[str(row["image"].resolve())] for row in group_rows}
                if len(group_splits) > 1:
                    duplicate_split_conflicts.append({
                        "sha256": digest, "splits": sorted(group_splits),
                        "images": [str(row["image"]) for row in group_rows],
                    })
            if not duplicate_split_conflicts:
                assignments = split_by_path
                split_frame = saved
                reused_existing_split = True
                print("Reusing saved split; exact duplicate groups are isolated by split.")
        else:
            print("Existing split has missing, extra, duplicate, or invalid rows; rebuilding it.")
    else:
        print("Existing split manifest has an invalid schema; rebuilding it.")

if duplicate_split_conflicts:
    print("Existing split leaks exact duplicate images across splits; rebuilding with hash groups.")
    legacy_root = PROJECT_ROOT / "archive" / "pre_duplicate_group_split"
    if legacy_root.exists():
        raise FileExistsError(f"Legacy archive already exists; inspect before moving artifacts: {legacy_root}")
    legacy_root.mkdir(parents=True)
    (legacy_root / "README.md").write_text(
        "# Archived pre-audit artifacts\n\n"
        "These generated data, checkpoints, and metrics came from a split that placed byte-identical raw images "
        "patches_101.jpg and patches_105.jpg in different splits (train and validation). The annotations differ "
        "slightly. These artifacts are preserved for provenance only and must not be used as final results. "
        "The raw dataset was not modified. The current pipeline groups exact SHA-256 duplicates before splitting.\n",
        encoding="utf-8",
    )
    def archive_move(source, relative_destination):
        source = Path(source)
        if not source.exists():
            return
        destination = legacy_root / relative_destination
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.exists():
            raise FileExistsError(destination)
        shutil.move(str(source), str(destination))

    for source, relative in [
        (DATA_ROOT / "processed", "data/processed"),
        (DATA_ROOT / "imbalance_variants", "data/imbalance_variants"),
        (SPLIT_MANIFEST, "data/splits/phase2_image_split.csv"),
        (RESULTS_ROOT / "phase2", "results/phase2"),
        (PROJECT_ROOT / "outputs" / "checkpoints", "outputs/checkpoints"),
        (PROJECT_ROOT / "outputs" / "logs", "outputs/logs"),
        (PROJECT_ROOT / "outputs" / "intermediate", "outputs/intermediate"),
        (PROJECT_ROOT / "outputs" / "plots", "outputs/plots"),
        (PROJECT_ROOT / "demo" / "predictions", "demo/predictions"),
    ]:
        archive_move(source, relative)
    for filename in [
        "phase3_baseline_metrics.json", "phase3_runtime.json", "phase4_summary.json",
        "phase4_final_distribution_report.csv", "phase5_summary.json",
        "phase5_method_distribution.csv", "phase5_validation_comparison.csv",
        "selected_model.json", "final_test_metrics.json",
    ]:
        archive_move(RESULTS_ROOT / filename, Path("results") / filename)
    print("Archived split-dependent generated artifacts at:", legacy_root)
    PHASE2_ROOT.mkdir(parents=True, exist_ok=True)

if not reused_existing_split:
    grouped_rows = list(image_hash_groups.items())
    group_presence = np.zeros((len(grouped_rows), len(CLASS_NAMES)), dtype=int)
    for group_index, (_, group_rows) in enumerate(grouped_rows):
        for row in group_rows:
            for class_id, _ in row["boxes"]:
                group_presence[group_index, class_id] = 1

    first = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
    train_group_idx, holdout_group_idx = next(first.split(np.zeros(len(grouped_rows)), group_presence))
    second = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
    val_rel, test_rel = next(second.split(
        np.zeros(len(holdout_group_idx)), group_presence[holdout_group_idx]
    ))
    group_splits = {}
    for i in train_group_idx:
        group_splits[grouped_rows[i][0]] = "train"
    for i in holdout_group_idx[val_rel]:
        group_splits[grouped_rows[i][0]] = "val"
    for i in holdout_group_idx[test_rel]:
        group_splits[grouped_rows[i][0]] = "test"

    manifest_rows = []
    for digest, group_rows in grouped_rows:
        split = group_splits[digest]
        for row in group_rows:
            source_key = str(row["image"].resolve())
            assignments[source_key] = split
            manifest_rows.append({
                "image_path": source_key, "split": split, "sha256": digest,
                "duplicate_group_id": digest[:16], "duplicate_group_size": len(group_rows),
            })
    split_frame = pd.DataFrame(manifest_rows).sort_values("image_path").reset_index(drop=True)
    SPLIT_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    split_frame.to_csv(SPLIT_MANIFEST, index=False)
    print("Created seeded group-aware multilabel split:", split_frame["split"].value_counts().to_dict(),
          "| unique image groups:", len(grouped_rows))
else:
    assignments = {str(Path(key).resolve()): value for key, value in assignments.items()}
    print("Reused image split:", pd.Series(assignments.values()).value_counts().to_dict())


Reusing saved split; exact duplicate groups are isolated by split.
Reused image split: {'train': 1258, 'test': 276, 'val': 266}


## 5. Convert to YOLO format or validate existing converted files

In [5]:
def yolo_row(class_id, coords, image_width, image_height):
    # Pascal VOC one-based inclusive -> zero-based half-open pixel box.
    x1 = coords["xmin"] - 1.0
    y1 = coords["ymin"] - 1.0
    x2 = coords["xmax"]
    y2 = coords["ymax"]
    bw, bh = x2 - x1, y2 - y1
    xc, yc = (x1 + x2) / 2.0, (y1 + y2) / 2.0
    values = (xc / image_width, yc / image_height, bw / image_width, bh / image_height)
    if not all(0 <= value <= 1 for value in values) or values[2] <= 0 or values[3] <= 0:
        raise ValueError(f"Converted coordinates outside normalized bounds: {values}")
    return f"{class_id} " + " ".join(f"{value:.8f}" for value in values)

expected_pairs = all(
    (PROCESSED_ROOT / "images" / split).is_dir()
    and (PROCESSED_ROOT / "labels" / split).is_dir()
    for split in ("train", "val", "test")
)
existing_manifest = pd.read_csv(CONVERSION_MANIFEST) if CONVERSION_MANIFEST.is_file() else pd.DataFrame()
manifest_complete = len(existing_manifest) == len(canonical_source_rows)
if manifest_complete:
    canonical_keys = {str(row["image"].resolve()) for row in canonical_source_rows}
    manifest_complete = set(existing_manifest["source_image"].map(lambda p: str(Path(p).resolve()))) == canonical_keys
if manifest_complete:
    manifest_complete = all(Path(p).is_file() for p in existing_manifest["yolo_image"]) and all(
        Path(p).is_file() for p in existing_manifest["yolo_label"]
    )
if manifest_complete:
    manifest_complete = all(
        str(Path(row.source_image).resolve()) in assignments
        and str(row.split) == assignments[str(Path(row.source_image).resolve())]
        for row in existing_manifest.itertuples(index=False)
    )

if REBUILD_PHASE2 or not manifest_complete or not expected_pairs:
    if PROCESSED_ROOT.exists() and any(PROCESSED_ROOT.rglob("*")):
        if not REBUILD_PHASE2:
            raise FileExistsError("Processed folders do not match the current split. Inspect them or set REBUILD_PHASE2=True to replace generated outputs.")
        archive_root = PROJECT_ROOT / "archive" / "manual_phase2_rebuild"
        if archive_root.exists():
            raise FileExistsError(f"Rebuild archive already exists; inspect it first: {archive_root}")
        archive_root.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(PROCESSED_ROOT), str(archive_root))
    manifest_rows = []
    for row in canonical_source_rows:
        source_key = str(row["image"].resolve())
        split = assignments[source_key]
        rel = row["image"].relative_to(RAW_DATA_ROOT).as_posix()
        suffix = hashlib.sha256(rel.encode("utf-8")).hexdigest()[:8]
        output_stem = f"{row['image'].stem}_{suffix}"
        image_out = PROCESSED_ROOT / "images" / split / f"{output_stem}{row['image'].suffix.lower()}"
        label_out = PROCESSED_ROOT / "labels" / split / f"{output_stem}.txt"
        image_out.parent.mkdir(parents=True, exist_ok=True)
        label_out.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(row["image"], image_out)
        label_lines = [yolo_row(class_id, coords, row["width"], row["height"]) for class_id, coords in row["boxes"]]
        label_out.write_text("\n".join(label_lines) + "\n", encoding="utf-8")
        manifest_rows.append({
            "source_image": str(row["image"]), "source_xml": str(row["xml"]), "split": split,
            "image_sha256": row["image_sha256"], "duplicate_group_id": row["image_sha256"][:16],
            "yolo_image": str(image_out), "yolo_label": str(label_out),
            "image_width": row["width"], "image_height": row["height"],
            "object_count": len(row["boxes"]),
            "classes_present": ",".join(sorted({CLASS_NAMES[cid] for cid, _ in row["boxes"]})),
        })
    conversion = pd.DataFrame(manifest_rows)
    conversion.to_csv(CONVERSION_MANIFEST, index=False)
else:
    conversion = existing_manifest
    print("Processed Phase 2 files already exist; preserving and validating them.")

def validate_yolo_file(path):
    errors = []
    for line_no, line in enumerate(Path(path).read_text(encoding="utf-8").splitlines(), start=1):
        fields = line.split()
        try:
            if len(fields) != 5: raise ValueError("expected 5 columns")
            class_id = int(fields[0])
            values = [float(x) for x in fields[1:]]
            if not 0 <= class_id < len(CLASS_NAMES): raise ValueError("class ID out of range")
            xc, yc, bw, bh = values
            if not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < bw <= 1 and 0 < bh <= 1): raise ValueError("values outside [0,1]")
            if xc - bw/2 < -1e-6 or xc + bw/2 > 1+1e-6 or yc - bh/2 < -1e-6 or yc + bh/2 > 1+1e-6:
                raise ValueError("box exceeds image boundary")
        except Exception as exc:
            errors.append(f"{Path(path).name}:{line_no}: {exc}")
    return errors

all_label_errors = []
for label_path in conversion["yolo_label"]:
    all_label_errors.extend(validate_yolo_file(label_path))
if all_label_errors:
    raise ValueError("\n".join(all_label_errors[:20]))
if len(conversion) != len(canonical_source_rows):
    raise ValueError(f"Expected {len(canonical_source_rows)} unique-content manifest rows, found {len(conversion)}")
print("YOLO rows validated:", len(conversion), "| malformed rows:", len(all_label_errors))


Processed Phase 2 files already exist; preserving and validating them.
YOLO rows validated: 1799 | malformed rows: 0


## 6. Write dataset YAML and summaries while keeping validation/test unchanged

In [6]:
from collections import Counter

class_count_rows = []
for class_id, class_name in enumerate(CLASS_NAMES):
    class_count_rows.append({
        "class_id": class_id, "class_name": class_name,
        "image_presence_count": sum(any(cid == class_id for cid, _ in row["boxes"]) for row in canonical_source_rows),
        "object_count": sum(sum(cid == class_id for cid, _ in row["boxes"]) for row in canonical_source_rows),
    })
class_counts_frame = pd.DataFrame(class_count_rows)
class_counts_frame.to_csv(PHASE2_ROOT / "class_counts_before_split.csv", index=False)

split_class_rows = []
for split_name in ("train", "val", "test"):
    split_rows = [row for row in canonical_source_rows
                  if assignments[str(row["image"].resolve())] == split_name]
    for class_id, class_name in enumerate(CLASS_NAMES):
        split_class_rows.append({
            "split": split_name, "class_id": class_id, "class_name": class_name,
            "images_with_class": sum(any(cid == class_id for cid, _ in row["boxes"]) for row in split_rows),
            "object_count": sum(sum(cid == class_id for cid, _ in row["boxes"]) for row in split_rows),
            "total_images": len(split_rows),
        })
pd.DataFrame(split_class_rows).to_csv(PHASE2_ROOT / "split_class_distribution.csv", index=False)

dataset_yaml = {
    "path": str(PROCESSED_ROOT.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {idx: name for idx, name in enumerate(CLASS_NAMES)},
}
DATA_YAML.write_text(yaml.safe_dump(dataset_yaml, sort_keys=False, allow_unicode=True), encoding="utf-8")

from datetime import datetime, timezone
prior_summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8")) if SUMMARY_PATH.is_file() else {}
class_counts_path = PHASE2_ROOT / "class_counts_before_split.csv"
class_counts_before_split = pd.read_csv(class_counts_path).to_dict(orient="records") if class_counts_path.is_file() else []
summary = {
    **prior_summary,
    "project": "Class-Imbalance-Aware Steel Surface Defect Detection Using Pretrained YOLO Models",
    "created_utc": prior_summary.get("created_utc", datetime.now(timezone.utc).isoformat()),
    "raw_dataset_directory": str(RAW_DATA_ROOT),
    "phase1_directory": str(DATA_ROOT),
    "coordinate_mode": COORDINATE_MODE,
    "class_names": CLASS_NAMES,
    "class_to_id": CLASS_TO_ID,
    "raw_image_count_matched": int(len(source_rows)),
    "exact_duplicate_group_count": int(len(duplicate_report)),
    "exact_duplicate_images_excluded_from_modeling": int(len(source_rows) - len(canonical_source_rows)),
    "duplicate_annotation_conflict_group_count": int((~duplicate_report["annotations_identical"]).sum()) if not duplicate_report.empty else 0,
    "image_count_matched": int(len(canonical_source_rows)),
    "image_count_converted": int(len(conversion)),
    "object_count_converted": int(conversion["object_count"].sum()),
    "split_method": "reused_existing_manifest" if reused_existing_split else "iterative_multilabel_stratification",
    "random_seed": SEED,
    "split_ratios_requested": prior_summary.get("split_ratios_requested", {"train": 0.70, "val": 0.15, "test": 0.15}),
    "split_image_counts": conversion["split"].value_counts().to_dict(),
    "class_counts_before_split": class_counts_before_split,
    "dataset_yaml": str(DATA_YAML),
    "split_manifest": str(SPLIT_MANIFEST),
    "conversion_manifest": str(CONVERSION_MANIFEST),
    "validation_status": "passed",
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
display(conversion.groupby("split").agg(images=("source_image","count"), objects=("object_count","sum")))
print("Dataset YAML:", DATA_YAML)
print("Summary:", SUMMARY_PATH)


,images,objects
split,,
test,276,640
train,1257,2915
val,266,631


Dataset YAML: ./configs/neu_det_yolo.yaml
Summary: ./results/phase2/phase2_dataset_summary.json
